In [1]:
import pandas as pd
from pathlib import Path
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Resolve project root (notebook lives in Model/)
ROOT = Path.cwd()
if not (ROOT / "jazz_harmony_ml_dataset.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "jazz_harmony_ml_dataset.csv"
data = pd.read_csv(DATA_PATH)

print(f"Loaded {len(data)} rows from {DATA_PATH}")
data.head()

/Users/zhaojinlan/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 816 rows from /Users/zhaojinlan/Documents/GitHub/AHM-Dataset/jazz_harmony_ml_dataset.csv


,Key,Chord_Progression,ChordName,Voicing,Emotion,Scale,File_Path
0,Bb,V7-#IVm7b5-IVm7-IIIm7,F7-Em7b5-Ebm7-Dm7,Four-Way Close,Joyful,Ionian,Dataset/Deceptive_Resolution/Primary/V7 - #IVm...
1,A,V7-#IVm7b5-IVm7-IIIm7,E7-Ebm7b5-Dm7-Dbm7,Four-Way Close,Joyful,Ionian,Dataset/Deceptive_Resolution/Primary/V7 - #IVm...
2,B,V7-#IVm7b5-IVm7-IIIm7,F#7-Fm7b5-Em7-Ebm7,Four-Way Close,Fantasy,Ionian,Dataset/Deceptive_Resolution/Primary/V7 - #IVm...
3,Db,V7-#IVm7b5-IVm7-IIIm7,Ab7-Gm7b5-F#m7-Fm7,Four-Way Close,Depressive,Ionian,Dataset/Deceptive_Resolution/Primary/V7 - #IVm...
4,C,V7-#IVm7b5-IVm7-IIIm7,G7-F#m7b5-Fm7-Em7,Four-Way Close,Joyful,Ionian,Dataset/Deceptive_Resolution/Primary/V7 - #IVm...


In [2]:
# Step 2: lookup tables — Emotion → Key, (Key, Progression) → ChordName

emotion_to_key = (
    data.drop_duplicates("Emotion")
    .set_index("Emotion")["Key"]
    .to_dict()
)

chordname_lookup = (
    data.drop_duplicates(["Key", "Chord_Progression"])
    .set_index(["Key", "Chord_Progression"])["ChordName"]
    .to_dict()
)

key_to_progressions = (
    data.groupby("Key")["Chord_Progression"]
    .apply(set)
    .to_dict()
)

assert len(emotion_to_key) == data["Emotion"].nunique()
assert data.groupby(["Key", "Chord_Progression"])["ChordName"].nunique().max() == 1

print(f"Emotion → Key mappings: {len(emotion_to_key)}")
print(f"(Key, Progression) → ChordName mappings: {len(chordname_lookup)}")
print("Example:", list(emotion_to_key.items())[:2])

Emotion → Key mappings: 13
(Key, Progression) → ChordName mappings: 204
Example: [('Joyful', 'Bb'), ('Fantasy', 'B')]


In [3]:
# Step 1: Emotion → Progression + Voicing (multi-task classification)

le_emotion = LabelEncoder()
le_progression = LabelEncoder()
le_voicing = LabelEncoder()

X = le_emotion.fit_transform(data["Emotion"]).reshape(-1, 1)
y_progression = le_progression.fit_transform(data["Chord_Progression"])
y_voicing = le_voicing.fit_transform(data["Voicing"])

split = train_test_split(
    X,
    y_progression,
    y_voicing,
    test_size=0.2,
    random_state=42,
    stratify=data["Emotion"],
)

X_train, X_test, yp_train, yp_test, yv_train, yv_test = split

N_TRIALS = 50
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 1.0]),
        "random_state": 42,
    }

    clf_p = RandomForestClassifier(**params)
    clf_v = RandomForestClassifier(**params)

    score_p = cross_val_score(
        clf_p, X_train, yp_train, cv=CV, scoring="neg_log_loss", n_jobs=-1
    ).mean()
    score_v = cross_val_score(
        clf_v, X_train, yv_train, cv=CV, scoring="neg_log_loss", n_jobs=-1
    ).mean()

    return (score_p + score_v) / 2


study = optuna.create_study(
    direction="maximize",
    study_name="emotion_harmony_rf",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = {**study.best_params, "random_state": 42}
print(f"Best CV neg_log_loss (avg): {study.best_value:.4f}")
print("Best params:", best_params)

clf_progression = RandomForestClassifier(**best_params)
clf_voicing = RandomForestClassifier(**best_params)

clf_progression.fit(X_train, yp_train)
clf_voicing.fit(X_train, yv_train)

print("\nTraining complete.")
print(f"  Progression classes: {len(le_progression.classes_)}")
print(f"  Voicing classes:     {len(le_voicing.classes_)}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: -2.28858:   0%|          | 0/50 [00:01<?, ?it/s]

Best trial: 0. Best value: -2.28858:   2%|▏         | 1/50 [00:01<01:30,  1.85s/it]

Best trial: 0. Best value: -2.28858:   2%|▏         | 1/50 [00:02<01:30,  1.85s/it]

Best trial: 0. Best value: -2.28858:   4%|▍         | 2/50 [00:02<01:03,  1.32s/it]

...

Best trial: 11. Best value: -2.0782:  98%|█████████▊| 49/50 [00:15<00:00,  4.89it/s]

Best trial: 11. Best value: -2.0782:  98%|█████████▊| 49/50 [00:15<00:00,  4.89it/s]

Best trial: 11. Best value: -2.0782: 100%|██████████| 50/50 [00:15<00:00,  4.30it/s]

Best trial: 11. Best value: -2.0782: 100%|██████████| 50/50 [00:15<00:00,  3.21it/s]

Best CV neg_log_loss (avg): -2.0782
Best params: {'n_estimators': 200, 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'random_state': 42}

Training complete.
  Progression classes: 17
  Voicing classes:     4


In [4]:
# Evaluation
# Accuracy is low by design: each Emotion has a uniform spread of Progression × Voicing.
# Sampling via predict_proba is the intended inference mode.

yp_pred = clf_progression.predict(X_test)
yv_pred = clf_voicing.predict(X_test)

prog_acc = accuracy_score(yp_test, yp_pred)
voicing_acc = accuracy_score(yv_test, yv_pred)
joint_acc = ((yp_pred == yp_test) & (yv_pred == yv_test)).mean()

# Top-3 accuracy (more meaningful with uniform labels)
def top_k_acc(clf, X, y_true, k=3):
    proba = clf.predict_proba(X)
    top_k = proba.argsort(axis=1)[:, -k:]
    return sum(int(y_true[i] in top_k[i]) for i in range(len(y_true))) / len(y_true)

print(f"Progression accuracy (argmax): {prog_acc:.3f}")
print(f"Voicing accuracy (argmax):     {voicing_acc:.3f}")
print(f"Joint (both correct):          {joint_acc:.3f}")
print(f"Progression top-3 accuracy:    {top_k_acc(clf_progression, X_test, yp_test, k=3):.3f}")
print(f"Voicing top-1 accuracy:        {top_k_acc(clf_voicing, X_test, yv_test, k=1):.3f}")

print("\n--- Progression report ---")
print(classification_report(
    yp_test, yp_pred, target_names=le_progression.classes_, zero_division=0
))

print("--- Voicing report ---")
print(classification_report(
    yv_test, yv_pred, target_names=le_voicing.classes_, zero_division=0
))

Progression accuracy (argmax): 0.037
Voicing accuracy (argmax):     0.146
Joint (both correct):          0.000
Progression top-3 accuracy:    0.159
Voicing top-1 accuracy:        0.146

--- Progression report ---
                       precision    recall  f1-score   support

        IIm7-V7-Imaj7       0.00      0.00      0.00        13
        IIm7b5-V7-Im7       0.20      0.20      0.20        10
           IVm6-Imaj7       0.00      0.00      0.00        10
          IVm7-V7-Im7       0.07      0.20      0.10        10
     IVm7-bVII7-Imaj7       0.00      0.00      0.00         9
    IVmaj7-IVm6-Imaj7       0.03      0.20      0.05         5
         IVmaj7-Imaj7       0.00      0.00      0.00         8
      IVmaj7-V7-Imaj7       0.00      0.00      0.00        10
V7-#IVm7b5-IVm7-IIIm7       0.00      0.00      0.00        15
             V7-IIIm7       0.00      0.00      0.00         9
              V7-VIm7       0.00      0.00      0.00         8
          V7-bIIImaj7       0.

In [5]:
import numpy as np


def _apply_temperature(probs: np.ndarray, temperature: float) -> np.ndarray:
    """Rescale probabilities by temperature.
    T→0: argmax; T=1: unchanged; T>1: flatter / more exploratory.
    """
    if temperature <= 0:
        out = np.zeros_like(probs, dtype=float)
        out[np.argmax(probs)] = 1.0
        return out
    if abs(temperature - 1.0) < 1e-9:
        return probs / probs.sum()
    scaled = np.power(probs, 1.0 / temperature)
    return scaled / scaled.sum()


def _sample_class(
    clf,
    le,
    x,
    valid_labels: set,
    rng: np.random.Generator,
    temperature: float,
    sample: bool,
) -> int:
    probs = clf.predict_proba(x)[0]
    classes = le.classes_
    valid_idx = [i for i, label in enumerate(classes) if label in valid_labels]
    valid_probs = probs[valid_idx]
    valid_probs = valid_probs / valid_probs.sum()

    if sample:
        scaled = _apply_temperature(valid_probs, temperature)
        local_idx = rng.choice(len(valid_idx), p=scaled)
        return valid_idx[local_idx]
    return valid_idx[int(np.argmax(valid_probs))]


def generate_harmony(
    emotion: str,
    sample: bool = True,
    temperature: float = 1.0,
    random_state: int = 42,
):
    """Predict Progression + Voicing from Emotion, then lookup ChordName.

    temperature (only when sample=True):
      - T → 0: mostly picks the top class
      - T = 1: uses model probabilities as-is
      - T > 1: more uniform / exploratory
    """
    if emotion not in emotion_to_key:
        raise ValueError(f"Unknown emotion: {emotion!r}")

    key = emotion_to_key[emotion]
    valid_progressions = key_to_progressions[key]
    valid_voicings = set(data.loc[data["Key"] == key, "Voicing"])
    x = le_emotion.transform([emotion]).reshape(-1, 1)
    rng = np.random.default_rng(random_state)

    prog_idx = _sample_class(
        clf_progression, le_progression, x, valid_progressions, rng, temperature, sample
    )
    voicing_idx = _sample_class(
        clf_voicing, le_voicing, x, valid_voicings, rng, temperature, sample
    )

    progression = le_progression.inverse_transform([prog_idx])[0]
    voicing = le_voicing.inverse_transform([voicing_idx])[0]
    chordname = chordname_lookup[(key, progression)]

    return {
        "Emotion": emotion,
        "Key": key,
        "Chord_Progression": progression,
        "Voicing": voicing,
        "ChordName": chordname,
    }


# Step 2 validation: lookup reproduces every ChordName in the dataset
lookup_ok = all(
    chordname_lookup[(row.Key, row.Chord_Progression)] == row.ChordName
    for row in data.itertuples()
)
print(f"ChordName lookup covers all rows: {lookup_ok}")

# Demo: deterministic prediction (argmax — one of many valid answers)
demo_emotion = "Joyful"
result = generate_harmony(demo_emotion, sample=False)
for k, v in result.items():
    print(f"  {k}: {v}")

ChordName lookup covers all rows: True
  Emotion: Joyful
  Key: Bb
  Chord_Progression: IVmaj7-IVm6-Imaj7
  Voicing: Four-Way Close
  ChordName: Ebmaj7-Ebm6-Bbmaj7


In [6]:
# Demo: temperature-controlled sampling
# T→0: conservative (top picks) | T=1: model distribution | T>1: more random

demo_emotion = "Depressive"
print(f"Sampled outputs for: {demo_emotion}\n")

for temp in [0.3, 1.0, 2.0]:
    print(f"Temperature = {temp}")
    for i in range(5):
        out = generate_harmony(
            demo_emotion, sample=True, temperature=temp, random_state=i
        )
        print(
            f"  [{i+1}] {out['Chord_Progression']} | "
            f"{out['Voicing']} → {out['ChordName']}"
        )
    print()

Sampled outputs for: Depressive

Temperature = 0.3
  [1] V7-bIIImaj7 | Drop 2+4 → Ab7-Emaj7
  [2] V7-IIIm7 | Four-Way Close → Ab7-Fm7
  [3] IVmaj7-Imaj7 | Drop 2+4 → F#maj7-Dbmaj7
  [4] IIm7-V7-Imaj7 | Drop 2+4 → Ebm7-Ab7-Dbmaj7
  [5] V7/II-bVImaj7 | Drop 2+4 → Bb7-Amaj7

Temperature = 1.0
  [1] V7-bIIImaj7 | Drop 2+4 → Ab7-Emaj7
  [2] V7-VIm7 | Four-Way Close → Ab7-Bbm7
  [3] IVmaj7-Imaj7 | Drop 2+4 → F#maj7-Dbmaj7
  [4] IVm7-bVII7-Imaj7 | Drop 2 → F#m7-B7-Dbmaj7
  [5] V7/II-bVImaj7 | Drop 3 → Bb7-Amaj7

Temperature = 2.0
  [1] V7-bIIImaj7 | Drop 2+4 → Ab7-Emaj7
  [2] V7-VIm7 | Four-Way Close → Ab7-Bbm7
  [3] IVmaj7-Imaj7 | Drop 2+4 → F#maj7-Dbmaj7
  [4] IVm7-bVII7-Imaj7 | Drop 2 → F#m7-B7-Dbmaj7
  [5] bVImaj7-bVII7-Imaj7 | Drop 3 → Amaj7-B7-Dbmaj7

